In [41]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import json 
import time 
import re
import random
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, construct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel

seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


In [42]:
def find_smaller_than_T(nums, T):
    for idx, num in enumerate(nums):
        if num < T:
            return True, idx
    return False, None

In [43]:
# Parameters
T = 0.6

# Ouput file
output_path = "reward/Llama-PRM800K/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_train_512_seed42_portion0.1.jsonl"

output_file = read_data(output_path)

print(f"Output file: {len(output_file)}")

Output file: 747


In [44]:
# first - verify
correct_correct = []
correct_incorrect = []
incorrect_correct = []
incorrect_incorrect = []

for i in tqdm(range(len(output_file))):
    true_answer = output_file[i]["answer"]
    pred_answer = output_file[i]["pred_ans"]
    step_probs = output_file[i]["step_probs"]
    
    vf, idx = find_smaller_than_T(step_probs, T)

    if vf:
        if pred_answer == true_answer:
            correct_incorrect.append(output_file[i])
        else:
            incorrect_incorrect.append(output_file[i])
    else:
        if pred_answer == true_answer:
            correct_correct.append(output_file[i])
        else:
            incorrect_correct.append(output_file[i])

print("[first - verify] results")
print(f"correct_correct: {len(correct_correct)}\nIndex: {[o['index'] for o in correct_correct]}\n")
print(f"correct_incorrect: {len(correct_incorrect)}\nIndex: {[o['index'] for o in correct_incorrect]}\n")
print(f"incorrect_correct: {len(incorrect_correct)}\nIndex: {[o['index'] for o in incorrect_correct]}\n")
print(f"incorrect_incorrect: {len(incorrect_incorrect)}\nIndex: {[o['index'] for o in incorrect_incorrect]}\n")
print(f"total num: {len(correct_correct) + len(correct_incorrect) + len(incorrect_correct) + len(incorrect_incorrect)}")

100%|██████████| 747/747 [00:00<00:00, 529874.02it/s]

[first - verify] results
correct_correct: 551
Index: [0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 20, 22, 23, 24, 25, 27, 28, 30, 31, 33, 35, 36, 38, 39, 40, 41, 42, 43, 44, 46, 49, 50, 51, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 81, 82, 83, 84, 86, 87, 88, 89, 90, 91, 93, 95, 96, 97, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 113, 115, 116, 117, 118, 119, 120, 122, 123, 125, 126, 127, 128, 130, 131, 133, 134, 135, 136, 139, 140, 143, 144, 145, 146, 147, 149, 152, 153, 154, 156, 161, 164, 165, 166, 168, 169, 170, 171, 174, 175, 176, 178, 179, 180, 181, 182, 184, 185, 186, 187, 188, 190, 191, 192, 193, 194, 196, 197, 198, 199, 200, 202, 204, 206, 207, 209, 210, 211, 212, 214, 215, 216, 217, 218, 221, 223, 224, 225, 226, 228, 230, 231, 232, 233, 234, 235, 236, 239, 241, 242, 243, 244, 247, 248, 250, 251, 252, 253, 254, 255, 256, 257, 258, 261, 262, 263, 264, 265, 267, 268, 270, 271, 273, 274, 275, 278

In [59]:
# Parameters
T = 0.6

# Ouput file
### LFF v11_5: cut comparison
output_path = "output/LFF_v11/GSM8K_Llama-3-8B-Instruct_LFF_v11_7_train_512_cut_seed42_portion0.1.jsonl"

output_file = read_data(output_path)

print(f"Output file: {len(output_file)}")

Output file: 747


In [51]:
def print_accuracy_table(a1, a2, a3, a4):
    print(f"{'first / verified':<15} | {'correct':^6} | {'incorrect':^6} |")
    print("-" * 33)
    print(f"{'correct':<15} | {a1:^6.2f} | {a2:^6.2f} |")
    print(f"{'incorrect':<15} | {a3:^6.2f} | {a4:^6.2f} |")

# 사용 예시
print_accuracy_table(0.91, 0.88, 0.94, 0.89)


first / verified | correct | incorrect |
---------------------------------
correct         |  0.91  |  0.88  |
incorrect       |  0.94  |  0.89  |


In [60]:
correct_correct = []
correct_correct_correct = []
correct_correct_incorrect = []

correct_incorrect = []
correct_incorrect_correct = []
correct_incorrect_incorrect = []

incorrect_correct = []
incorrect_correct_correct = []
incorrect_correct_incorrect = []

incorrect_incorrect = []
incorrect_incorrect_correct = []
incorrect_incorrect_incorrect = []

for i in tqdm(range(len(output_file))):
    true_answer = output_file[i]["answer"]
    pred_answer = output_file[i]["pred_ans"]
    step_probs1 = output_file[i]["step_probs1"]
    step_probs2 = output_file[i]["step_probs2"]
    
    vf1, idx1 = find_smaller_than_T(step_probs1, T)
    vf2, idx2 = find_smaller_than_T(step_probs2, T)

    if vf1:
        if pred_answer == true_answer:
            correct_incorrect.append(output_file[i])
            if vf2:
                correct_incorrect_incorrect.append(output_file[i])
            else: 
                correct_incorrect_correct.append(output_file[i])
        else:
            incorrect_incorrect.append(output_file[i])
            if vf2:
                incorrect_incorrect_incorrect.append(output_file[i])
            else:
                incorrect_incorrect_correct.append(output_file[i])
    else:
        if pred_answer == true_answer:
            correct_correct.append(output_file[i])
            if vf2:
                correct_correct_incorrect.append(output_file[i])
            else:
                correct_correct_correct.append(output_file[i])
        else:
            incorrect_correct.append(output_file[i])
            if vf2:
                incorrect_correct_incorrect.append(output_file[i])
            else:
                incorrect_correct_correct.append(output_file[i])

print(f"correct_correct: {len(correct_correct)}\nIndex: {[o['index'] for o in correct_correct]}")
print(f"\tcorrect_correct_correct: {len(correct_correct_correct)}\n\tIndex: {[o['index'] for o in correct_correct_correct]}\n")
print(f"\tcorrect_correct_incorrect: {len(correct_correct_incorrect)}\n\tIndex: {[o['index'] for o in correct_correct_incorrect]}")

print(f"correct_incorrect: {len(correct_incorrect)}\nIndex: {[o['index'] for o in correct_incorrect]}")
print(f"\tcorrect_incorrect_correct: {len(correct_incorrect_correct)}\n\tIndex: {[o['index'] for o in correct_incorrect_correct]}\n")
print(f"\tcorrect_incorrect_incorrect: {len(correct_incorrect_incorrect)}\n\tIndex: {[o['index'] for o in correct_incorrect_incorrect]}")

print(f"incorrect_correct: {len(incorrect_correct)}\nIndex: {[o['index'] for o in incorrect_correct]}")
print(f"\tincorrect_correct_correct: {len(incorrect_correct_correct)}\n\tIndex: {[o['index'] for o in incorrect_correct_correct]}\n")
print(f"\tincorrect_correct_incorrect: {len(incorrect_correct_incorrect)}\n\tIndex: {[o['index'] for o in incorrect_correct_incorrect]}")

print(f"incorrect_incorrect: {len(incorrect_incorrect)}\nIndex: {[o['index'] for o in incorrect_incorrect]}")
print(f"\tincorrect_incorrect_correct: {len(incorrect_incorrect_correct)}\n\tIndex: {[o['index'] for o in incorrect_incorrect_correct]}\n")
print(f"\tincorrect_incorrect_incorrect: {len(incorrect_incorrect_incorrect)}\n\tIndex: {[o['index'] for o in incorrect_incorrect_incorrect]}")

print(f"total num: {len(correct_correct) + len(correct_incorrect) + len(incorrect_correct) + len(incorrect_incorrect)}")

print()
print("original")
print_accuracy_table(len(correct_correct), len(correct_incorrect), len(incorrect_correct), len(incorrect_incorrect))

print()
print("modified")
print_accuracy_table(len(correct_correct_correct) + len(correct_incorrect_correct), len(correct_correct_incorrect) + len(correct_incorrect_incorrect), len(incorrect_correct_correct) + len(incorrect_incorrect_correct), len(incorrect_correct_incorrect) + len(incorrect_incorrect_incorrect))

100%|██████████| 747/747 [00:00<00:00, 292107.50it/s]

correct_correct: 559
Index: [0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 20, 22, 23, 24, 25, 27, 28, 30, 31, 33, 35, 36, 38, 39, 40, 41, 42, 43, 44, 46, 48, 49, 50, 51, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 81, 82, 83, 84, 86, 87, 88, 89, 90, 91, 93, 95, 96, 97, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 113, 115, 116, 117, 118, 119, 120, 122, 123, 125, 126, 127, 128, 130, 131, 133, 134, 135, 136, 139, 140, 143, 144, 145, 146, 147, 149, 152, 153, 154, 156, 161, 164, 165, 166, 168, 169, 170, 171, 174, 175, 176, 178, 179, 180, 181, 182, 184, 185, 186, 187, 188, 190, 191, 192, 193, 194, 196, 197, 198, 199, 200, 202, 204, 206, 207, 209, 210, 211, 212, 214, 215, 216, 217, 218, 221, 223, 224, 225, 226, 228, 230, 231, 232, 233, 234, 235, 236, 239, 241, 242, 243, 244, 247, 248, 250, 251, 252, 253, 254, 255, 256, 257, 258, 261, 262, 263, 264, 265, 267, 268, 270, 271, 273, 274, 275, 278, 279, 280, 281, 283,